## 1. 라이브러리 및 환경 설정

In [1]:
import os
import pandas as pd
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
import timm

# 하이퍼파라미터 설정
CFG = {
    'IMG_SIZE': 384,
    'EPOCHS': 30,
    'LEARNING_RATE': 3e-4,      # freeze 중 새 레이어 빠르게 학습
    'FINETUNE_LR': 5e-5,        # unfreeze 후 전체 fine-tune LR
    'WEIGHT_DECAY': 1e-4,
    'BATCH_SIZE': 16,
    'SEED': 42,
    'PATIENCE': 10,
    'FREEZE_EPOCHS': 5,         # B4에서 3이 부족했으므로 5로 확대
    'LABEL_SMOOTHING': 0.05,
    'TTA_NUM': 10
}

def seed_everything(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(CFG['SEED'])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

c:\Users\user\anaconda3\envs\a\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. 데이터 로드 및 학습/검증 데이터 분할

In [2]:
#train_df = pd.read_csv('./data/train.csv')
train_df = pd.read_csv('./data/train_physics.csv')
val_df = pd.read_csv('./data/dev.csv')

print(f"학습 데이터 개수: {len(train_df)}")
print(f"검증 데이터 개수: {len(val_df)}")

학습 데이터 개수: 1000
검증 데이터 개수: 100


## 3. 커스텀 데이터셋 클래스 정의 및 데이터 로더

In [3]:
import torch
from torch.utils.data import Dataset
from PIL import Image
import os
import torchvision.transforms.functional as F

class MultiViewDataset(Dataset):
    def __init__(self, df, root_dir, transform=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform
        self.is_test = is_test
        self.label_map = {'stable': 0, 'unstable': 1}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        sample_id = str(row['id'])
        folder_path = os.path.join(self.root_dir, sample_id)

        views = []
        for name in ["front", "top"]:
            img_path = os.path.join(folder_path, f"{name}.png")
            # 1. PIL Image로 로드
            image = Image.open(img_path).convert("RGB")
            
            # 2. [중요] Numpy 에러를 피하기 위해 F.to_tensor를 직접 호출
            # 이 함수는 PIL을 직접 받아서 float32 텐서(0~1)로 변환해줍니다.
            image_tensor = F.to_tensor(image) 
            
            # 3. Transform 적용 
            if self.transform:
                # 주의: 이미 텐서이므로 transform 리스트에 ToTensor()가 있으면 에러가 날 수 있음 (아래 2번 참고)
                image_tensor = self.transform(image_tensor)
            
            views.append(image_tensor.float())
            
        if self.is_test:
            return views
        
        label = self.label_map[row['label']]
        combined_lean = float(row.get('combined_lean', 0.0))
        
        return views, label, torch.tensor([combined_lean], dtype=torch.float32)

In [10]:
# 수정된 train_transform: ToTensor()를 제거하고 텐서 입력 기반으로 재구성
train_transform = transforms.Compose([
    # transforms.ToTensor(),  <-- Dataset에서 처리하므로 삭제
    transforms.Resize((420, 420)),
    transforms.RandomCrop(CFG['IMG_SIZE']),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.02),
    # RandomAffine, GaussianBlur 등은 텐서 입력도 잘 지원함
    transforms.RandomAffine(degrees=5, translate=(0.05, 0.05), scale=(0.95, 1.05)),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.1, scale=(0.02, 0.1)),
])

# test_transform과 tta_transform에서도 ToTensor()를 삭제하세요.
test_transform = transforms.Compose([
    transforms.Resize((CFG['IMG_SIZE'], CFG['IMG_SIZE'])),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ─── 수정 후 ───
tta_transform = transforms.Compose([
    transforms.Resize((420, 420)),
    transforms.RandomCrop(CFG['IMG_SIZE']),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.05),
    # ToTensor() 삭제됨 (Dataset 내부에서 이미 처리함)
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = MultiViewDataset(train_df, './data/train', train_transform, is_test=False)
val_dataset = MultiViewDataset(val_df, './data/dev', test_transform, is_test=False)

train_loader = DataLoader(train_dataset, batch_size=CFG['BATCH_SIZE'], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=CFG['BATCH_SIZE'], shuffle=False)

test_df = pd.read_csv('./data/sample_submission.csv')
test_dataset = MultiViewDataset(test_df, './data/test', test_transform, is_test=True)
test_loader = DataLoader(test_dataset, batch_size=CFG['BATCH_SIZE'], shuffle=False)

## 4. 모델 정의 (EfficientNet-V2-S + Cross-View Attention)

In [5]:
# ─── Building Blocks ─────────────────────────────────────────────────

class CrossViewAttention(nn.Module):
    """Bidirectional cross-attention between front and top view features."""
    def __init__(self, embed_dim, num_heads=4, dropout=0.1):
        super().__init__()
        self.cross_attn_f2t = nn.MultiheadAttention(
            embed_dim, num_heads, dropout=dropout, batch_first=True
        )
        self.cross_attn_t2f = nn.MultiheadAttention(
            embed_dim, num_heads, dropout=dropout, batch_first=True
        )
        self.norm_f = nn.LayerNorm(embed_dim)
        self.norm_t = nn.LayerNorm(embed_dim)
        self.ffn_f = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(embed_dim * 4, embed_dim), nn.Dropout(dropout),
        )
        self.ffn_t = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(embed_dim * 4, embed_dim), nn.Dropout(dropout),
        )
        self.norm_ffn_f = nn.LayerNorm(embed_dim)
        self.norm_ffn_t = nn.LayerNorm(embed_dim)

    def forward(self, feat_front, feat_top):
        attn_f, _ = self.cross_attn_f2t(feat_front, feat_top, feat_top)
        feat_front = self.norm_f(feat_front + attn_f)
        attn_t, _ = self.cross_attn_t2f(feat_top, feat_front, feat_front)
        feat_top = self.norm_t(feat_top + attn_t)
        feat_front = self.norm_ffn_f(feat_front + self.ffn_f(feat_front))
        feat_top = self.norm_ffn_t(feat_top + self.ffn_t(feat_top))
        return feat_front, feat_top


class ViewDiscrepancyModule(nn.Module):
    """Encodes cross-view discrepancy as an explicit anomaly signal."""
    def __init__(self, feat_dim):
        super().__init__()
        self.fusion = nn.Sequential(
            nn.Linear(feat_dim * 4, feat_dim), nn.GELU(),
            nn.Dropout(0.1), nn.Linear(feat_dim, feat_dim),
        )

    def forward(self, feat_front, feat_top):
        diff = feat_front - feat_top
        hadamard = feat_front * feat_top
        combined = torch.cat([feat_front, feat_top, diff, hadamard], dim=1)
        return self.fusion(combined)


# ─── Main Model ──────────────────────────────────────────────────────
class CollapsePredictor(nn.Module):
    """
    v5-Lite: EfficientNet-V2-S + Cross-View Attention + Physics Regression
    - Task 1: Stability Classification (Stable/Unstable)
    - Task 2: Physics Regression (combined_lean) -> 핵심 보조 태스크
    """
    def __init__(self, num_classes=1, project_dim=128, num_attn_layers=1, num_heads=4):
        super().__init__()

        # --- Backbone 및 기존 모듈 유지 ---
        self.backbone = timm.create_model(
            'tf_efficientnetv2_s.in21k_ft_in1k',
            pretrained=True,
            features_only=True,
            out_indices=[4]
        )
        backbone_dim = self.backbone.feature_info.channels()[-1]
        
        self.proj = nn.Sequential(
            nn.Conv2d(backbone_dim, project_dim, 1),
            nn.BatchNorm2d(project_dim),
            nn.GELU(),
        )
        
        self.view_embed = nn.Parameter(torch.randn(2, 1, project_dim) * 0.02)

        self.cross_attn_layers = nn.ModuleList([
            CrossViewAttention(project_dim, num_heads=num_heads)
            for _ in range(num_attn_layers)
        ])

        self.discrepancy = ViewDiscrepancyModule(project_dim)

        # --- Shared Layer (특징 융합 강화) ---
        # combined 차원이 project_dim * 5 이므로 이를 처리할 공통 bottleneck 생성
        self.bottleneck = nn.Sequential(
            nn.Linear(project_dim * 5, project_dim),
            nn.GELU(),
            nn.Dropout(0.4),
        )

        # --- Multi-Task Heads ---
        # 1. Classification Head (기존 목적)
        self.classifier = nn.Linear(project_dim, num_classes)
        
        # 2. Physics Regression Head (combined_lean 예측)
        # 물리 수치는 스칼라 값이므로 출력 차원은 1입니다.
        self.regressor = nn.Sequential(
            nn.Linear(project_dim, 64),
            nn.GELU(),
            nn.Linear(64, 1)
        )

    def _extract(self, x):
        feat_map = self.backbone(x)[-1]
        feat_map = self.proj(feat_map)
        
        B, C, H, W = feat_map.shape
        spatial = feat_map.view(B, C, H * W).permute(0, 2, 1)
        global_feat = feat_map.mean(dim=[2, 3])
        return spatial, global_feat

    def forward(self, views):
        front_img, top_img = views[0], views[1]

        # 1. Feature Extraction
        spatial_f, global_f = self._extract(front_img)
        spatial_t, global_t = self._extract(top_img)

        # 2. View Embedding & Cross-Attention
        spatial_f = spatial_f + self.view_embed[0]
        spatial_t = spatial_t + self.view_embed[1]

        sf, st = spatial_f, spatial_t
        for attn_layer in self.cross_attn_layers:
            sf, st = attn_layer(sf, st)

        cross_f = sf.mean(dim=1)
        cross_t = st.mean(dim=1)

        # 3. Discrepancy & Combine
        disc = self.discrepancy(global_f, global_t)
        combined = torch.cat([cross_f, cross_t, disc, global_f, global_t], dim=1)
        
        # 4. Shared Bottleneck
        shared_feat = self.bottleneck(combined)

        # 5. Multi-Task Outputs
        logits = self.classifier(shared_feat)    # For Stable/Unstable classification
        physics = self.regressor(shared_feat)   # For combined_lean regression
        
        return logits, physics


# ── 모델 정보 출력 ────────────────────────────────────────────────────
_m = CollapsePredictor().to(device)
total_params = sum(p.numel() for p in _m.parameters()) / 1e6
backbone_params = sum(p.numel() for p in _m.backbone.parameters()) / 1e6
new_params = total_params - backbone_params
print(f'Total: {total_params:.1f}M | Backbone(frozen): {backbone_params:.1f}M | New layers: {new_params:.1f}M')

# backbone feature 확인
with torch.no_grad():
    dummy = torch.randn(1, 3, 384, 384).to(device)
    feat = _m.backbone(dummy)[-1]
    print(f'Backbone output: {feat.shape}')  # [1, 256, 12, 12]
del _m

Unexpected keys (bn2.bias, bn2.num_batches_tracked, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


Total: 20.4M | Backbone(frozen): 19.8M | New layers: 0.6M
Backbone output: torch.Size([1, 256, 12, 12])


## 4. 학습 및 검증 루프 구현

In [6]:
def train_one_epoch(model, loader, criterion_cls, criterion_reg, optimizer, device):
    model.train()
    train_loss = 0
    
    # LAMBDA_PHYSICS: 물리 학습의 비중 (분석 결과가 좋으므로 5.0~10.0 추천)
    lambda_physics = CFG.get('LAMBDA_PHYSICS', 5.0) 
    
    for views, labels, physics_gt in tqdm(loader, desc="Training"):
        views = [v.to(device) for v in views]
        labels = labels.to(device).float().view(-1, 1)
        physics_gt = physics_gt.to(device).float().view(-1, 1)
        
        # Label Smoothing 적용
        smooth = CFG['LABEL_SMOOTHING']
        labels_smoothed = labels * (1 - smooth) + (0.5 * smooth) # 0.5는 이진 분류의 중간값
        
        optimizer.zero_grad()
        
        # 모델의 Multi-task 출력을 받음
        logits, pred_physics = model(views)
        
        # 1. 분류 손실 (LogLoss 최적화)
        loss_cls = criterion_cls(logits, labels_smoothed)
        
        # 2. 물리 회귀 손실 (MSE)
        loss_reg = criterion_reg(pred_physics, physics_gt)
        
        # 3. 통합 손실 (물리 수치를 맞추려 노력할수록 분류 성능이 오름)
        loss = loss_cls + (lambda_physics * loss_reg)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item()
        
    return train_loss / len(loader)

def validate(model, loader, device):
    model.eval()
    all_probs = []
    all_labels = []
    
    with torch.no_grad():
        for views, labels, _ in tqdm(loader, desc="Validation"): # physics_gt는 사용 안함
            views = [v.to(device) for v in views]
            labels = labels.to(device).float()
            
            # 모델에서 logits만 가져옴 (두 번째 반환값인 physics는 무시)
            logits, _ = model(views)
            probs = torch.sigmoid(logits).view(-1)
            
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    all_probs = np.array(all_probs, dtype=np.float64)
    all_labels = np.array(all_labels, dtype=np.float64)
    
    eps = 1e-15
    p = np.clip(all_probs, eps, 1 - eps)
    # 공식 대회 지표: LogLoss
    logloss_score = -np.mean(all_labels * np.log(p) + (1 - all_labels) * np.log(1 - p))
    acc_score = np.mean((all_probs > 0.5) == all_labels)
    
    return logloss_score, acc_score

In [7]:
model = CollapsePredictor().to(device)

# Multi-task를 위한 두 가지 Criterion 정의
criterion_cls = nn.BCEWithLogitsLoss()
criterion_reg = nn.MSELoss() 

# ── Phase 1: Backbone Freeze (5 epochs) ──
for param in model.backbone.parameters():
    param.requires_grad = False

# 새로 추가된 bottleneck과 regressor를 포함하여 optimizer 설정
optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CFG['LEARNING_RATE'],       
    weight_decay=CFG['WEIGHT_DECAY']
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CFG['FREEZE_EPOCHS'], eta_min=1e-5
)

# ── Training Loop ──
best_logloss = float('inf')
patience_counter = 0

for epoch in range(1, CFG['EPOCHS'] + 1):
    # Phase 2: Unfreeze backbone
    if epoch == CFG['FREEZE_EPOCHS'] + 1:
        print('\n── Unfreezing EfficientNet-V2-S backbone ──')
        for param in model.backbone.parameters():
            param.requires_grad = True
            
        # 모든 신규 레이어(proj, attn, disc, bottleneck, classifier, regressor) 업데이트
        optimizer = optim.AdamW([
            {'params': model.backbone.parameters(), 'lr': CFG['FINETUNE_LR'] * 0.1},
            {'params': model.proj.parameters(),      'lr': CFG['FINETUNE_LR']},
            {'params': model.cross_attn_layers.parameters(), 'lr': CFG['FINETUNE_LR']},
            {'params': model.discrepancy.parameters(),       'lr': CFG['FINETUNE_LR']},
            {'params': model.bottleneck.parameters(),        'lr': CFG['FINETUNE_LR']}, # 추가
            {'params': model.classifier.parameters(),        'lr': CFG['FINETUNE_LR']},
            {'params': model.regressor.parameters(),         'lr': CFG['FINETUNE_LR']}, # 추가
            {'params': [model.view_embed],                   'lr': CFG['FINETUNE_LR']},
        ], weight_decay=CFG['WEIGHT_DECAY'])
        
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=CFG['EPOCHS'] - CFG['FREEZE_EPOCHS'], eta_min=1e-6
        )

    # 수정된 train_one_epoch 호출 (criterion 두 개 전달)
    avg_train_loss = train_one_epoch(model, train_loader, criterion_cls, criterion_reg, optimizer, device)
    
    # 수정된 validate 호출 (criterion 생략 가능하도록 설계했으므로 맞춰서 호출)
    val_logloss, val_acc = validate(model, val_loader, device)
    
    scheduler.step()

    current_lr = optimizer.param_groups[0]['lr']
    print(f'Epoch [{epoch}] lr={current_lr:.6f}')
    print(f'  - Train Loss: {avg_train_loss:.4f}')
    print(f'  - Val Log-Loss: {val_logloss:.6f} | Val Acc: {val_acc:.4f}')

    if val_logloss < best_logloss:
        best_logloss = val_logloss
        patience_counter = 0
        torch.save(model.state_dict(), 'best_model.pt')
        print(f'  ★ Best model saved (Log-Loss: {best_logloss:.6f})')
    else:
        patience_counter += 1
        print(f'  - No improvement ({patience_counter}/{CFG["PATIENCE"]})')

    if patience_counter >= CFG['PATIENCE']:
        print(f'\nEarly stopping at epoch {epoch}')
        break

model.load_state_dict(torch.load('best_model.pt', weights_only=True))
print(f'\nBest Val Log-Loss: {best_logloss:.6f}')

Unexpected keys (bn2.bias, bn2.num_batches_tracked, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.
Validation: 100%|██████████| 7/7 [00:01<00:00,  5.70it/s]


Epoch [1] lr=0.000272
  - Train Loss: 1.3448
  - Val Log-Loss: 0.668654 | Val Acc: 0.5600
  ★ Best model saved (Log-Loss: 0.668654)


Validation: 100%|██████████| 7/7 [00:01<00:00,  5.75it/s]


Epoch [2] lr=0.000200
  - Train Loss: 0.8465
  - Val Log-Loss: 0.653706 | Val Acc: 0.5700
  ★ Best model saved (Log-Loss: 0.653706)


Validation: 100%|██████████| 7/7 [00:01<00:00,  6.26it/s]


...

Epoch [17] lr=0.000003
  - Train Loss: 0.3034
  - Val Log-Loss: 0.449808 | Val Acc: 0.8300
  - No improvement (8/10)


Validation: 100%|██████████| 7/7 [00:00<00:00,  7.36it/s]


Epoch [18] lr=0.000003
  - Train Loss: 0.3005
  - Val Log-Loss: 0.444021 | Val Acc: 0.8000
  - No improvement (9/10)


Validation: 100%|██████████| 7/7 [00:00<00:00,  7.24it/s]

Epoch [19] lr=0.000003
  - Train Loss: 0.3020
  - Val Log-Loss: 0.474185 | Val Acc: 0.7900
  - No improvement (10/10)

Early stopping at epoch 19

Best Val Log-Loss: 0.429771


## 5. 추론 (TTA: Test Time Augmentation)

In [11]:
model.eval()

# ── 1) 원본(clean) 예측 ──────────────────────────────────────────────
clean_probs = []
with torch.no_grad():
    for views in tqdm(test_loader, desc="Clean inference"):
        views = [v.to(device) for v in views]
        
        # [수정] 튜플 반환 처리: logits만 가져오고 physics는 무시(_)
        logits, _ = model(views) 
        logits = logits.view(-1)
        
        probs = torch.sigmoid(logits)
        clean_probs.extend(probs.cpu().numpy())
clean_probs = np.array(clean_probs, dtype=np.float64)

# ── 2) TTA 예측 ──────────────────────────────────────────────────────
tta_probs_list = []

for tta_i in range(CFG['TTA_NUM']):
    # TTA용 데이터셋은 기존 MultiViewDataset(또는 PhysicsInformedDataset의 is_test=True 모드) 사용
    tta_dataset = MultiViewDataset(test_df, './data/test', tta_transform, is_test=True)
    tta_loader = DataLoader(tta_dataset, batch_size=CFG['BATCH_SIZE'], shuffle=False)
    
    round_probs = []
    with torch.no_grad():
        for views in tta_loader:
            views = [v.to(device) for v in views]
            
            # [수정] 튜플 반환 처리
            logits, _ = model(views)
            logits = logits.view(-1)
            
            probs = torch.sigmoid(logits)
            round_probs.extend(probs.cpu().numpy())
    
    tta_probs_list.append(np.array(round_probs, dtype=np.float64))
    print(f'  TTA round {tta_i+1}/{CFG["TTA_NUM"]} done')

# ── 3) 가중 평균 ─────────────────────────────────────────────────────
all_rounds = [clean_probs] + tta_probs_list
weights = [2.0] + [1.0] * CFG['TTA_NUM']
weights = np.array(weights) / sum(weights)

all_probs = np.zeros_like(clean_probs)
for w, p in zip(weights, all_rounds):
    all_probs += w * p

print(f'\nFinal prediction stats: mean={all_probs.mean():.4f}, std={all_probs.std():.4f}')
print(f'  min={all_probs.min():.4f}, max={all_probs.max():.4f}')

# ── 4) 결과 저장 ─────────────────────────────────────────────────────
submission = pd.DataFrame({
    'id': test_df['id'],
    'unstable_prob': all_probs,
    'stable_prob': 1.0 - all_probs
})

os.makedirs("results", exist_ok=True)
submission.to_csv('./results/submission.csv', encoding='UTF-8-sig', index=False)
print("submission.csv 저장 완료. (EfficientNet-V2-S + TTA)")

Clean inference: 100%|██████████| 63/63 [00:10<00:00,  5.77it/s]


  TTA round 1/10 done
  TTA round 2/10 done
  TTA round 3/10 done
  TTA round 4/10 done
  TTA round 5/10 done
  TTA round 6/10 done
  TTA round 7/10 done
  TTA round 8/10 done
  TTA round 9/10 done
  TTA round 10/10 done

Final prediction stats: mean=0.5586, std=0.2882
  min=0.0345, max=0.9804
submission.csv 저장 완료. (EfficientNet-V2-S + TTA)
